# CAM Rows from Saved Llama-70B Residual Streams + Hamming WTA Classifiers

This version performs **no model loading and no activation extraction**. It reads the residual-stream cache already produced by the original notebook and uses it for CAM-row extraction, classifier training, and cross-domain evaluation.

Two full-dimensional binary experiments are evaluated:

1. **Direct sign hash** — hash both each residual stream and each stored CAM vector using the sign of every original residual dimension.
2. **Learned full hash matrix** — learn a dense shared matrix \(H \in \mathbb{R}^{d \times d}\), multiply both residual streams and CAM vectors by \(H\), and then hash by sign.

Both methods use normalized Hamming similarity

\[
\operatorname{sim}_H(a,b)=1-2\frac{d_H(a,b)}{d}=\frac{1}{d}\sum_k a_kb_k,\qquad a_k,b_k\in\{-1,+1\},
\]

followed by learned positive scalar weights for the CAM rows and a WTA maximum. The learned hash matrix is trained through the sign operation using a straight-through estimator.

There is **no down-projection, no Euclidean-distance path, and no LSQ quantization** in this notebook.

> Memory note: for an 8192-dimensional residual stream, a dense 8192×8192 fp32 matrix contains about 67 million parameters and occupies about 256 MiB before gradients and optimizer state.


In [ ]:
# Optional installs. Run only in a fresh environment.
%pip install -q torch scikit-learn pandas numpy tqdm matplotlib seaborn


## 1. Configuration


In [ ]:
from pathlib import Path
import os
import json
import math
import random
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

# ---- Metadata describing the already-saved residual streams ----
MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"
LAYER_INDEX = 33
HIDDEN_STATE_MODE = "post_layer"

# ---- Paths ----
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "cam_probe_wta_outputs_llama70b_L33"
ARTIFACT_DIR = OUTPUT_DIR / "hamming_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# This is the mmap directory created by the original notebook. It can instead be:
#   1. a directory containing manifest.json and X_mean.npy/y_mean.npy/etc.,
#   2. a legacy .npz activation cache, or
#   3. a .pt/.pth dictionary containing the same arrays.
DEFAULT_SAVED_RESIDUAL_PATH = (
    OUTPUT_DIR
    / "activation_cache"
    / f"{MODEL_ID.replace('/', '__')}_L{LAYER_INDEX}_{HIDDEN_STATE_MODE}_activations_mmap"
)
SAVED_RESIDUAL_PATH = Path(
    os.environ.get("SAVED_RESIDUAL_PATH", str(DEFAULT_SAVED_RESIDUAL_PATH))
).expanduser()

# ---- Preferred domain order; domains actually present in the cache are detected later ----
PREFERRED_DOMAINS = [
    "definitional",
    "empirical",
    "fictional",
    "logical",
    "ethical",
    "sycophancy",
    "exp_inverted",
    "roleplaying",
    "insider_trading",
    "sandbagging",
    "truthfulqa",
    "fever",
]

# The current notebook's cache was produced after the empirical-label fix, so this
# is empty by default. For a cache saved before that fix, set this to {"empirical"}.
CACHED_LABEL_FLIP_DOMAINS = set()

# ---- Shared split ----
SHARED_TEST_FRACTION = 0.20
SHARED_VAL_FRACTION_OF_TRAIN = 0.20
RANDOM_SEED = 0

# ---- CAM direction extraction ----
N_DIRECTIONS_PER_SINGLE_DOMAIN = 4
N_DIRECTIONS_COMBINED = 10
LOGREG_ALPHA = 1e-4
USE_STANDARD_SCALER_FOR_PROBES = False
REMOVE_GENERAL_BEFORE_SINGLE_DOMAIN = False
POSITIVE_LABEL_NAME = "deceptive_or_false"

# Use example-level averaged residual streams by default. "tokens" is supported
# only when the saved cache contains nonempty X_token arrays.
TRAIN_WTA_ON = "means"

# ---- Hash experiments ----
HASH_METHODS = ["sign", "learned_matrix"]
HASH_METHOD_TITLES = {
    "sign": "Direct sign hash",
    "learned_matrix": "Learned full d × d hash matrix",
}

# The code length is always the original embedding dimension d.
FIXED_HASH_EPOCHS = 150
LEARNED_HASH_EPOCHS = 30
HASH_EARLY_STOPPING_PATIENCE = 6
FIXED_HASH_EVAL_EVERY = 5
LEARNED_HASH_EVAL_EVERY = 2

HASH_READOUT_LR = 3e-2
HASH_MATRIX_LR = 1e-4
HASH_READOUT_WEIGHT_DECAY = 1e-4
HASH_MATRIX_WEIGHT_DECAY = 1e-6

FIXED_HASH_BATCH_SIZE = 512
LEARNED_HASH_BATCH_SIZE = 32
FIXED_HASH_EVAL_BATCH_SIZE = 2048
LEARNED_HASH_EVAL_BATCH_SIZE = 64

# Regularizers for the learned dense hash matrix.
# The identity penalty is computed without allocating a second d × d identity tensor.
HASH_IDENTITY_REG = 1e-4
HASH_BIT_BALANCE_REG = 1e-3
HASH_STE_TEMPERATURE = 1.0

# Threshold is fitted on validation examples only. AUROC does not use a threshold.
THRESHOLD_SELECTION_METRIC = "accuracy"  # "accuracy" or "f1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available()

# Saving every learned 8192×8192 matrix can consume several GB. Set True only when needed.
SAVE_LEARNED_HASH_MATRICES = False

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Saved residual path:", SAVED_RESIDUAL_PATH)
print("Device:", DEVICE, "| bf16 autocast:", USE_BF16)
print("Hash methods:", HASH_METHODS)


## 2. Load saved residual streams only


In [ ]:
def load_saved_residual_streams(path: Path):
    """Load the residual-stream cache without loading an LLM or extracting activations."""
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(
            f"Saved residual cache not found at {path}. "
            "Edit SAVED_RESIDUAL_PATH or set the SAVED_RESIDUAL_PATH environment variable."
        )

    if path.is_dir():
        manifest_path = path / "manifest.json"
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            names = list(manifest.get("arrays", {}).keys())
        else:
            names = [p.stem for p in sorted(path.glob("*.npy"))]

        if not names:
            raise ValueError(f"No .npy arrays found in residual cache directory: {path}")

        out = {}
        for name in names:
            arr_path = path / f"{name}.npy"
            if not arr_path.exists():
                raise FileNotFoundError(arr_path)
            # Residual matrices remain memory-mapped; metadata arrays are small enough to load.
            mmap_mode = "r" if name.startswith("X_") else None
            out[name] = np.load(arr_path, allow_pickle=False, mmap_mode=mmap_mode)
        source_type = "mmap directory"

    elif path.suffix.lower() == ".npz":
        obj = np.load(path, allow_pickle=True)
        out = {k: obj[k] for k in obj.files}
        source_type = "npz"

    elif path.suffix.lower() in {".pt", ".pth"}:
        obj = torch.load(path, map_location="cpu")
        if not isinstance(obj, dict):
            raise TypeError("A .pt/.pth residual cache must contain a dictionary of arrays.")
        out = {
            k: (v.detach().cpu().numpy() if torch.is_tensor(v) else np.asarray(v))
            for k, v in obj.items()
        }
        source_type = "torch dictionary"

    else:
        raise ValueError(
            f"Unsupported residual cache format: {path}. Use a cache directory, .npz, .pt, or .pth."
        )

    required_mean = {"X_mean", "y_mean", "domain_mean", "example_index_mean"}
    missing = sorted(required_mean.difference(out))
    if missing:
        raise KeyError(f"Saved residual cache is missing required arrays: {missing}")

    print(f"Loaded saved residual streams from {source_type}: {path}")
    print({k: tuple(v.shape) for k, v in out.items() if hasattr(v, "shape")})
    return out


def optionally_flip_cached_labels(acts_obj, domains_to_flip):
    """Apply an explicit label flip only for caches known to predate a label fix."""
    if not domains_to_flip:
        return acts_obj

    out = dict(acts_obj)
    for suffix in ["mean", "token"]:
        y_key = f"y_{suffix}"
        d_key = f"domain_{suffix}"
        if y_key not in out or d_key not in out:
            continue
        labels = np.asarray(out[y_key], dtype=np.int64).copy()
        domains = np.asarray(out[d_key]).astype(str)
        mask = np.isin(domains, sorted(domains_to_flip))
        labels[mask] = 1 - labels[mask]
        out[y_key] = labels
        print(f"Flipped {int(mask.sum())} cached {suffix} labels for {sorted(domains_to_flip)}")
    return out


acts = load_saved_residual_streams(SAVED_RESIDUAL_PATH)
acts = optionally_flip_cached_labels(acts, CACHED_LABEL_FLIP_DOMAINS)

available_domains = list(dict.fromkeys(np.asarray(acts["domain_mean"]).astype(str).tolist()))
DOMAINS = [d for d in PREFERRED_DOMAINS if d in available_domains]
DOMAINS += [d for d in available_domains if d not in DOMAINS]

if TRAIN_WTA_ON == "tokens":
    required_token = {"X_token", "y_token", "domain_token", "example_index_token"}
    missing_token = sorted(required_token.difference(acts))
    if missing_token or len(acts.get("X_token", [])) == 0:
        raise ValueError(
            "TRAIN_WTA_ON='tokens' was requested, but the saved cache has no token residuals. "
            f"Missing/empty token arrays: {missing_token or ['X_token']}"
        )

D_MODEL = int(acts["X_mean"].shape[1])
print("Detected residual dimension / binary code length:", D_MODEL)
print("Domains in saved residual cache:", DOMAINS)

if D_MODEL >= 4096:
    n_params = D_MODEL * D_MODEL
    print(
        f"A full learned {D_MODEL}×{D_MODEL} hash matrix has {n_params:,} parameters "
        f"({n_params * 4 / 2**20:.1f} MiB in fp32 before gradients/optimizer state)."
    )


## 3. Shared split and CAM-row extraction


In [ ]:
def l2_normalize_rows(X, eps=1e-8):
    X = np.asarray(X, dtype=np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def l2_normalize_vec(w, eps=1e-8):
    w = np.asarray(w, dtype=np.float32)
    return w / (np.linalg.norm(w) + eps)


def project_out_directions(X, directions):
    if directions is None or len(directions) == 0:
        return X
    D = l2_normalize_rows(np.asarray(directions, dtype=np.float32))
    Q = []
    for d in D:
        v = d.copy()
        for q in Q:
            v = v - np.dot(v, q) * q
        n = np.linalg.norm(v)
        if n > 1e-6:
            Q.append(v / n)
    if not Q:
        return X
    Q = np.stack(Q)
    return X - (X @ Q.T) @ Q


def fit_logreg_direction(X, y, alpha=LOGREG_ALPHA, use_scaler=USE_STANDARD_SCALER_FOR_PROBES):
    if len(np.unique(y)) < 2:
        raise ValueError("Need both classes to fit a direction.")

    if use_scaler:
        scaler = StandardScaler(with_mean=True, with_std=True)
        X_fit = scaler.fit_transform(X)
    else:
        scaler = None
        X_fit = X

    clf = LogisticRegression(
        C=1.0 / alpha,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1,
    )
    clf.fit(X_fit, y)

    w_scaled = clf.coef_[0].astype(np.float32)
    b_scaled = float(clf.intercept_[0])

    if scaler is not None:
        scale = scaler.scale_.astype(np.float32)
        mean = scaler.mean_.astype(np.float32)
        denom = np.maximum(scale, 1e-12)
        w = w_scaled / denom
        b = b_scaled - float(np.dot(w_scaled / denom, mean))
    else:
        w, b = w_scaled, b_scaled

    return l2_normalize_vec(w), b, clf


def extract_inlp_directions(X, y, n_directions, name_prefix, alpha=LOGREG_ALPHA):
    X_work = np.asarray(X, dtype=np.float32).copy()
    rows = []
    metadata = []

    for k in range(n_directions):
        try:
            w, b, _ = fit_logreg_direction(X_work, y, alpha=alpha)
        except Exception as exc:
            print(f"[{name_prefix}] stopping at k={k}: {exc}")
            break

        rows.append(w)
        metadata.append({
            "row_name": f"{name_prefix}__inlp_{k}",
            "scope": name_prefix,
            "k": k,
            "bias_from_probe": b,
            "alpha": alpha,
            "positive_label": POSITIVE_LABEL_NAME,
            "split_source": "shared_train_examples",
        })
        X_work = project_out_directions(X_work, [w])

    if not rows:
        raise RuntimeError(f"No CAM directions were extracted for {name_prefix}")
    return np.stack(rows).astype(np.float32), metadata


def get_scope_mask(domains_array, scope):
    domains_array = np.asarray(domains_array).astype(str)
    if scope == "combined":
        return np.ones(len(domains_array), dtype=bool)
    return domains_array == scope


def _split_stratify_labels(df):
    return (df["domain"].astype(str) + "__label_" + df["label"].astype(str)).to_numpy()


def _safe_stratify_or_none(labels, test_size, split_name):
    labels = np.asarray(labels)
    counts = pd.Series(labels).value_counts()
    if len(counts) < 2:
        print(f"[{split_name}] Only one stratum; using an unstratified split.")
        return None

    n_test = int(np.ceil(len(labels) * test_size)) if isinstance(test_size, float) else int(test_size)
    n_train = len(labels) - n_test
    if counts.min() < 2 or n_test < len(counts) or n_train < len(counts):
        print(f"[{split_name}] Strata are too small; using an unstratified split.")
        return None
    return labels


# One example-level split is reused by INLP extraction and both hash experiments.
ex_df = pd.DataFrame({
    "ex_idx": np.asarray(acts["example_index_mean"], dtype=np.int64),
    "domain": np.asarray(acts["domain_mean"]).astype(str),
    "label": np.asarray(acts["y_mean"], dtype=np.int64),
}).drop_duplicates("ex_idx").reset_index(drop=True)

all_positions = np.arange(len(ex_df))
strat_labels = _split_stratify_labels(ex_df)
trainval_pos, test_pos = train_test_split(
    all_positions,
    test_size=SHARED_TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=_safe_stratify_or_none(strat_labels, SHARED_TEST_FRACTION, "shared train/test"),
)

trainval_df = ex_df.iloc[trainval_pos].reset_index(drop=True)
trainval_positions = np.arange(len(trainval_df))
trainval_strat_labels = _split_stratify_labels(trainval_df)
train_pos_local, val_pos_local = train_test_split(
    trainval_positions,
    test_size=SHARED_VAL_FRACTION_OF_TRAIN,
    random_state=RANDOM_SEED,
    stratify=_safe_stratify_or_none(
        trainval_strat_labels,
        SHARED_VAL_FRACTION_OF_TRAIN,
        "shared train/val",
    ),
)

GLOBAL_SPLIT_EXAMPLE_IDS = {
    "train": np.sort(trainval_df.iloc[train_pos_local]["ex_idx"].to_numpy()),
    "val": np.sort(trainval_df.iloc[val_pos_local]["ex_idx"].to_numpy()),
    "test": np.sort(ex_df.iloc[test_pos]["ex_idx"].to_numpy()),
}

_split_sets = {k: set(map(int, v)) for k, v in GLOBAL_SPLIT_EXAMPLE_IDS.items()}
assert _split_sets["train"].isdisjoint(_split_sets["val"])
assert _split_sets["train"].isdisjoint(_split_sets["test"])
assert _split_sets["val"].isdisjoint(_split_sets["test"])


def _mask_for_split(example_indices, split_name):
    return np.isin(example_indices, GLOBAL_SPLIT_EXAMPLE_IDS[split_name])


split_summary = []
for split_name in GLOBAL_SPLIT_EXAMPLE_IDS:
    mask = _mask_for_split(acts["example_index_mean"], split_name)
    split_summary.append(pd.DataFrame({
        "split": split_name,
        "domain": np.asarray(acts["domain_mean"])[mask],
        "label": np.asarray(acts["y_mean"])[mask],
    }))
split_summary = pd.concat(split_summary, ignore_index=True)

print("Shared example split sizes:", {k: len(v) for k, v in GLOBAL_SPLIT_EXAMPLE_IDS.items()})
display(split_summary.groupby(["split", "domain", "label"]).size().rename("n").reset_index())

train_mean_mask = _mask_for_split(acts["example_index_mean"], "train")
X_mean = np.asarray(acts["X_mean"][train_mean_mask], dtype=np.float32)
y_mean = np.asarray(acts["y_mean"][train_mean_mask], dtype=np.int64)
domain_mean = np.asarray(acts["domain_mean"][train_mean_mask]).astype(str)

print("Shared-train residual matrix for INLP:", X_mean.shape)


In [ ]:
# Combined/general CAM rows.
combined_rows, combined_meta = extract_inlp_directions(
    X_mean,
    y_mean,
    n_directions=N_DIRECTIONS_COMBINED,
    name_prefix="combined",
)
print("Combined rows:", combined_rows.shape)

all_scope_rows = {"combined": combined_rows}
all_scope_meta = {"combined": combined_meta}

# Single-domain CAM rows.
for domain in DOMAINS:
    mask = get_scope_mask(domain_mean, domain)
    if mask.sum() == 0:
        continue
    Xd = X_mean[mask]
    yd = y_mean[mask]
    if len(np.unique(yd)) < 2:
        print(f"Skipping {domain}: only one class in shared train split")
        continue
    if REMOVE_GENERAL_BEFORE_SINGLE_DOMAIN:
        Xd = project_out_directions(Xd, combined_rows)

    rows, meta = extract_inlp_directions(
        Xd,
        yd,
        n_directions=N_DIRECTIONS_PER_SINGLE_DOMAIN,
        name_prefix=domain,
    )
    all_scope_rows[domain] = rows
    all_scope_meta[domain] = meta
    print(f"{domain:20s}", rows.shape, "labels", dict(zip(*np.unique(yd, return_counts=True))))

scopes_to_train = [s for s in ["combined"] + DOMAINS if s in all_scope_rows]
print("Scopes available for hash experiments:", scopes_to_train)


## 4. Direct sign-hash sanity check


In [ ]:
def binary_sign_np(x):
    x = np.asarray(x)
    return np.where(x >= 0, 1, -1).astype(np.int8)


def hamming_similarity_np(X, rows, batch_size=512):
    """Return max normalized Hamming similarity and winning CAM row."""
    row_codes = binary_sign_np(rows).astype(np.float32)
    all_scores, all_winners = [], []
    d = row_codes.shape[1]

    for start in range(0, len(X), batch_size):
        x_codes = binary_sign_np(np.asarray(X[start:start + batch_size], dtype=np.float32)).astype(np.float32)
        # mean(bit_product) = 1 - 2 * normalized Hamming distance
        similarity = (x_codes @ row_codes.T) / d
        all_scores.append(similarity.max(axis=1))
        all_winners.append(similarity.argmax(axis=1))

    return np.concatenate(all_scores), np.concatenate(all_winners)


def safe_auroc(y, scores):
    y = np.asarray(y, dtype=np.int64)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, scores))


sign_sanity_rows = []
for scope, rows in all_scope_rows.items():
    mask = get_scope_mask(domain_mean, scope)
    scores, _ = hamming_similarity_np(X_mean[mask], rows)
    sign_sanity_rows.append({
        "scope": scope,
        "n": int(mask.sum()),
        "raw_sign_hash_train_auroc": safe_auroc(y_mean[mask], scores),
    })

display(pd.DataFrame(sign_sanity_rows).sort_values("scope"))


## 5. Hamming WTA models and training


In [ ]:
def hard_binary_sign(z):
    return torch.where(z >= 0, torch.ones_like(z), -torch.ones_like(z))


def straight_through_binary_sign(z, temperature=HASH_STE_TEMPERATURE, eps=1e-6):
    """Hard ±1 forward pass with a scaled tanh surrogate gradient."""
    hard = hard_binary_sign(z)
    surrogate_scale = z.detach().abs().mean(dim=-1, keepdim=True).clamp_min(eps)
    soft = torch.tanh(z / (temperature * surrogate_scale))
    return soft + (hard - soft).detach()


class HammingWTAClassifier(nn.Module):
    """WTA classifier whose row comparisons depend only on full-length binary codes."""

    def __init__(self, cam_rows, method):
        super().__init__()
        if method not in {"sign", "learned_matrix"}:
            raise ValueError(method)

        rows = torch.as_tensor(np.asarray(cam_rows, dtype=np.float32))
        self.register_buffer("cam_rows", rows)
        self.method = method
        self.d_model = int(rows.shape[1])
        self.n_rows = int(rows.shape[0])

        if method == "learned_matrix":
            self.hash_matrix = nn.Linear(self.d_model, self.d_model, bias=False)
            # Identity initialization makes epoch 0 exactly the direct-sign experiment.
            with torch.no_grad():
                self.hash_matrix.weight.zero_()
                self.hash_matrix.weight.diagonal().fill_(1.0)
        else:
            self.hash_matrix = None

        # softplus(raw_alpha) keeps row weights positive.
        initial_raw_alpha = math.log(math.expm1(1.0))
        self.raw_alphas = nn.Parameter(torch.full((self.n_rows,), initial_raw_alpha))
        self.bias = nn.Parameter(torch.zeros(()))

    @property
    def alphas(self):
        return F.softplus(self.raw_alphas) + 1e-6

    def transform(self, z):
        if self.hash_matrix is None:
            return z
        return self.hash_matrix(z)

    def encode(self, z):
        transformed = self.transform(z)
        if self.method == "learned_matrix" and self.training:
            return straight_through_binary_sign(transformed)
        return hard_binary_sign(transformed)

    def forward(self, x):
        query_codes = self.encode(x)
        row_codes = self.encode(self.cam_rows)

        # The float32 accumulation gives stable Hamming counts even when H is evaluated in bf16.
        hamming_similarity = (query_codes.float() @ row_codes.float().T) / self.d_model
        normalized_hamming = 0.5 * (1.0 - hamming_similarity)

        row_scores = hamming_similarity * self.alphas.unsqueeze(0)
        logits, winners = row_scores.max(dim=1)
        logits = logits + self.bias

        return logits, winners, {
            "normalized_hamming": normalized_hamming,
            "query_codes": query_codes,
            "row_codes": row_codes,
        }

    def identity_penalty(self):
        if self.hash_matrix is None:
            return self.bias.new_zeros(())
        W = self.hash_matrix.weight
        d = self.d_model
        # ||W-I||_F^2 / d^2 without constructing an additional identity matrix.
        return (W.square().sum() - 2.0 * W.diagonal().sum() + d) / (d * d)

    @torch.no_grad()
    def export_small(self):
        was_training = self.training
        self.eval()
        row_codes = self.encode(self.cam_rows).cpu().numpy().astype(np.int8)
        out = {
            "method": self.method,
            "alphas": self.alphas.detach().cpu().numpy().astype(np.float32),
            "bias": float(self.bias.detach().cpu()),
            "cam_binary_codes": row_codes,
            "d_model": self.d_model,
        }
        self.train(was_training)
        return out


def make_train_data_for_scope(scope, acts_obj=None, train_on=TRAIN_WTA_ON, split=None, return_example_ids=False):
    if acts_obj is None:
        acts_obj = acts

    if train_on == "means":
        X = acts_obj["X_mean"]
        y = acts_obj["y_mean"]
        domains = acts_obj["domain_mean"]
        example_ids = acts_obj["example_index_mean"]
    elif train_on == "tokens":
        X = acts_obj["X_token"]
        y = acts_obj["y_token"]
        domains = acts_obj["domain_token"]
        example_ids = acts_obj["example_index_token"]
    else:
        raise ValueError(train_on)

    mask = get_scope_mask(domains, scope)
    if split is not None:
        if split not in GLOBAL_SPLIT_EXAMPLE_IDS:
            raise ValueError(f"Unknown split: {split}")
        mask = mask & np.isin(example_ids, GLOBAL_SPLIT_EXAMPLE_IDS[split])

    Xs = X[mask]
    ys = np.asarray(y[mask], dtype=np.int64)
    exs = np.asarray(example_ids[mask], dtype=np.int64)
    if return_example_ids:
        return Xs, ys, exs
    return Xs, ys


def make_hash_split_indices(example_ids):
    example_ids = np.asarray(example_ids, dtype=np.int64)
    return {
        split_name: np.flatnonzero(np.isin(example_ids, ids)).astype(np.int64)
        for split_name, ids in GLOBAL_SPLIT_EXAMPLE_IDS.items()
    }


def split_label_counts(y, split_indices):
    out = {}
    for split_name, idxs in split_indices.items():
        labels, counts = np.unique(y[idxs], return_counts=True) if len(idxs) else ([], [])
        out[split_name] = {int(k): int(v) for k, v in zip(labels, counts)}
    return out


def method_batch_size(method, evaluation=False):
    if method == "sign":
        return FIXED_HASH_EVAL_BATCH_SIZE if evaluation else FIXED_HASH_BATCH_SIZE
    return LEARNED_HASH_EVAL_BATCH_SIZE if evaluation else LEARNED_HASH_BATCH_SIZE


def score_dataset_batched(model, X, method, return_winners=False):
    model.eval()
    scores, winners = [], []
    batch_size = method_batch_size(method, evaluation=True)

    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb_np = np.asarray(X[start:start + batch_size], dtype=np.float32)
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=DEVICE)
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=USE_BF16 and DEVICE == "cuda" and method == "learned_matrix",
            ):
                logits, batch_winners, _ = model(xb)
            scores.append(logits.float().cpu().numpy())
            if return_winners:
                winners.append(batch_winners.cpu().numpy())

    score_array = np.concatenate(scores) if scores else np.array([], dtype=np.float32)
    if return_winners:
        winner_array = np.concatenate(winners) if winners else np.array([], dtype=np.int64)
        return score_array, winner_array
    return score_array


def select_validation_threshold(y, scores, metric=THRESHOLD_SELECTION_METRIC):
    y = np.asarray(y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)
    if len(scores) == 0:
        return 0.0

    candidates = np.unique(np.quantile(scores, np.linspace(0.0, 1.0, 201)))
    best_value, best_threshold = -np.inf, float(np.median(scores))

    for threshold in candidates:
        pred = (scores >= threshold).astype(np.int64)
        if metric == "accuracy":
            value = accuracy_score(y, pred)
        elif metric == "f1":
            _, _, value, _ = precision_recall_fscore_support(
                y, pred, average="binary", zero_division=0
            )
        else:
            raise ValueError(metric)

        if value > best_value:
            best_value = value
            best_threshold = float(threshold)

    return best_threshold


def metrics_from_scores(y, scores, threshold):
    y = np.asarray(y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)
    pred = (scores >= threshold).astype(np.int64)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auroc": safe_auroc(y, scores),
        "threshold": float(threshold),
        "n": int(len(y)),
    }


def train_hamming_wta(cam_rows, X, y, split_indices, method, seed=RANDOM_SEED):
    if len(np.unique(y[split_indices["train"]])) < 2:
        raise ValueError("Training split must contain both labels.")

    train_idx = np.asarray(split_indices["train"], dtype=np.int64)
    val_idx = np.asarray(split_indices["val"], dtype=np.int64)
    test_idx = np.asarray(split_indices["test"], dtype=np.int64)

    model = HammingWTAClassifier(cam_rows, method=method).to(DEVICE)

    readout_params = [model.raw_alphas, model.bias]
    if method == "learned_matrix":
        optimizer = torch.optim.AdamW([
            {
                "params": readout_params,
                "lr": HASH_READOUT_LR,
                "weight_decay": HASH_READOUT_WEIGHT_DECAY,
            },
            {
                "params": model.hash_matrix.parameters(),
                "lr": HASH_MATRIX_LR,
                "weight_decay": HASH_MATRIX_WEIGHT_DECAY,
            },
        ])
        epochs = LEARNED_HASH_EPOCHS
        eval_every = LEARNED_HASH_EVAL_EVERY
    else:
        optimizer = torch.optim.AdamW(
            readout_params,
            lr=HASH_READOUT_LR,
            weight_decay=HASH_READOUT_WEIGHT_DECAY,
        )
        epochs = FIXED_HASH_EPOCHS
        eval_every = FIXED_HASH_EVAL_EVERY

    y_train = np.asarray(y[train_idx], dtype=np.float32)
    positive_count = max(float(y_train.sum()), 1.0)
    negative_count = max(float(len(y_train) - y_train.sum()), 1.0)
    pos_weight = torch.tensor(negative_count / positive_count, dtype=torch.float32, device=DEVICE)

    best_state = None
    best_val_auc = -np.inf
    evaluations_without_improvement = 0
    history = []
    rng = np.random.default_rng(seed)

    for epoch in range(epochs):
        model.train()
        permutation = rng.permutation(len(train_idx))
        epoch_losses = []
        batch_size = method_batch_size(method, evaluation=False)

        for start in range(0, len(permutation), batch_size):
            local = permutation[start:start + batch_size]
            batch_idx = train_idx[local]

            xb = torch.as_tensor(
                np.asarray(X[batch_idx], dtype=np.float32),
                dtype=torch.float32,
                device=DEVICE,
            )
            yb = torch.as_tensor(y_train[local], dtype=torch.float32, device=DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=USE_BF16 and DEVICE == "cuda" and method == "learned_matrix",
            ):
                logits, _, extra = model(xb)
                bce = F.binary_cross_entropy_with_logits(logits.float(), yb, pos_weight=pos_weight)

                if method == "learned_matrix":
                    identity_penalty = model.identity_penalty().float()
                    balance_penalty = extra["query_codes"].float().mean(dim=0).square().mean()
                    loss = (
                        bce
                        + HASH_IDENTITY_REG * identity_penalty
                        + HASH_BIT_BALANCE_REG * balance_penalty
                    )
                else:
                    identity_penalty = bce.new_zeros(())
                    balance_penalty = bce.new_zeros(())
                    loss = bce

            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))

            del xb, yb, logits, extra, loss

        should_evaluate = ((epoch + 1) % eval_every == 0) or (epoch == epochs - 1)
        if not should_evaluate:
            continue

        val_scores = score_dataset_batched(model, X[val_idx], method)
        val_auc = safe_auroc(y[val_idx], val_scores)
        history.append({
            "epoch": int(epoch + 1),
            "loss": float(np.mean(epoch_losses)),
            "val_auroc": float(val_auc),
            "identity_penalty": float(identity_penalty.detach().cpu()),
            "bit_balance_penalty": float(balance_penalty.detach().cpu()),
        })

        print(
            f"[{method}] epoch {epoch + 1:3d}/{epochs} "
            f"loss={np.mean(epoch_losses):.4f} val_AUROC={val_auc:.4f}"
        )

        if not np.isnan(val_auc) and val_auc > best_val_auc + 1e-5:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            evaluations_without_improvement = 0
        else:
            evaluations_without_improvement += 1

        if evaluations_without_improvement >= HASH_EARLY_STOPPING_PATIENCE:
            print(f"[{method}] early stopping after epoch {epoch + 1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    val_scores = score_dataset_batched(model, X[val_idx], method)
    threshold = select_validation_threshold(y[val_idx], val_scores)
    test_scores = score_dataset_batched(model, X[test_idx], method)

    val_metrics = metrics_from_scores(y[val_idx], val_scores, threshold)
    val_metrics.update({"split": "val", "source_split": "shared_val"})
    test_metrics = metrics_from_scores(y[test_idx], test_scores, threshold)
    test_metrics.update({"split": "test", "source_split": "shared_test"})

    artifact = model.export_small()
    artifact.update({
        "threshold": float(threshold),
        "history": history,
        "metrics": [val_metrics, test_metrics],
        "best_val_auroc": float(best_val_auc),
    })
    return model, artifact


## 6. Train both hash experiments and cross-evaluate


In [ ]:
hash_metrics_rows = []
hash_artifacts = {method: {} for method in HASH_METHODS}

for method_index, method in enumerate(HASH_METHODS):
    print("\n" + "=" * 100)
    print(HASH_METHOD_TITLES[method])
    print("=" * 100)

    for scope_index, train_scope in enumerate(scopes_to_train):
        rows = all_scope_rows[train_scope]
        X_scope, y_scope, example_ids = make_train_data_for_scope(
            train_scope,
            split=None,
            return_example_ids=True,
        )
        split_indices = make_hash_split_indices(example_ids)

        if len(np.unique(y_scope[split_indices["train"]])) < 2:
            print(f"Skipping {train_scope}: shared train split has one class")
            continue

        print(
            f"\nTraining method={method}, train_scope={train_scope}, "
            f"CAM rows={rows.shape}, residuals={X_scope.shape}, "
            f"split counts={split_label_counts(y_scope, split_indices)}"
        )

        model, artifact = train_hamming_wta(
            rows,
            X_scope,
            y_scope,
            split_indices,
            method=method,
            seed=RANDOM_SEED + 1000 * method_index + scope_index,
        )

        # Same-domain validation and test metrics.
        for metric_row in artifact["metrics"]:
            row = dict(metric_row)
            row.update({
                "method": method,
                "method_title": HASH_METHOD_TITLES[method],
                "train_scope": train_scope,
                "eval_scope": train_scope,
                "n_rows": int(rows.shape[0]),
                "code_length": int(D_MODEL),
                "train_on": TRAIN_WTA_ON,
            })
            hash_metrics_rows.append(row)

        # Cross-domain evaluation uses the source model's validation-selected threshold.
        threshold = artifact["threshold"]
        for eval_scope in scopes_to_train:
            if eval_scope == train_scope:
                continue
            X_eval, y_eval = make_train_data_for_scope(eval_scope, split="test")
            if len(y_eval) == 0 or len(np.unique(y_eval)) < 2:
                continue

            scores = score_dataset_batched(model, X_eval, method)
            row = metrics_from_scores(y_eval, scores, threshold)
            row.update({
                "split": "cross_eval",
                "source_split": "shared_test",
                "method": method,
                "method_title": HASH_METHOD_TITLES[method],
                "train_scope": train_scope,
                "eval_scope": eval_scope,
                "n_rows": int(rows.shape[0]),
                "code_length": int(D_MODEL),
                "train_on": TRAIN_WTA_ON,
            })
            hash_metrics_rows.append(row)

        # Save the learned full matrix only when explicitly enabled.
        matrix_path = None
        if method == "learned_matrix" and SAVE_LEARNED_HASH_MATRICES:
            matrix_path = ARTIFACT_DIR / (
                f"hash_matrix_{train_scope}_L{LAYER_INDEX}_{HIDDEN_STATE_MODE}_{D_MODEL}x{D_MODEL}.pt"
            )
            torch.save(model.hash_matrix.weight.detach().cpu(), matrix_path)
            print("Saved learned hash matrix:", matrix_path)

        artifact["hash_matrix_path"] = str(matrix_path) if matrix_path else None
        hash_artifacts[method][train_scope] = artifact

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

hash_metrics_df = pd.DataFrame(hash_metrics_rows)
metrics_df = hash_metrics_df  # convenient compatibility alias for later analysis cells

display(
    hash_metrics_df
    .sort_values(["method", "train_scope", "eval_scope", "split"])
    .style.format({
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
        "auroc": "{:.3f}",
        "threshold": "{:.3f}",
    })
)


## 7. AUROC heatmap for each hashing experiment


In [ ]:
heatmap_df = hash_metrics_df[
    hash_metrics_df["split"].isin(["test", "cross_eval"])
].copy()

for method in HASH_METHODS:
    method_df = heatmap_df[heatmap_df["method"] == method]
    if method_df.empty:
        print("No results for", method)
        continue

    pivot_auroc = method_df.pivot(
        index="train_scope",
        columns="eval_scope",
        values="auroc",
    )

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        pivot_auroc,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        cbar_kws={"label": "AUROC"},
    )
    plt.title(
        f"Cross-Domain Hamming WTA AUROC\n{HASH_METHOD_TITLES[method]}"
    )
    plt.xlabel("Evaluated On")
    plt.ylabel("Trained On")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# Accuracy is threshold-dependent, so it is shown as a table using the threshold
# selected only on the source-domain validation split.
same_domain_results = heatmap_df[
    heatmap_df["train_scope"] == heatmap_df["eval_scope"]
][["method_title", "train_scope", "accuracy", "auroc", "n"]]

print("Same-domain held-out accuracy and AUROC:")
display(
    same_domain_results
    .sort_values(["method_title", "train_scope"])
    .style.format({"accuracy": "{:.3f}", "auroc": "{:.3f}"})
)


## 8. Save compact results and binary CAM codes


In [ ]:
metrics_path = ARTIFACT_DIR / (
    f"hamming_metrics_L{LAYER_INDEX}_{HIDDEN_STATE_MODE}_code{D_MODEL}.csv"
)
hash_metrics_df.to_csv(metrics_path, index=False)
print("Saved metrics:", metrics_path)

for method, scope_artifacts in hash_artifacts.items():
    for scope, artifact in scope_artifacts.items():
        out_path = ARTIFACT_DIR / (
            f"cam_hamming_{method}_{scope}_L{LAYER_INDEX}_{HIDDEN_STATE_MODE}_code{D_MODEL}.npz"
        )
        np.savez_compressed(
            out_path,
            cam_rows=np.asarray(all_scope_rows[scope], dtype=np.float32),
            cam_binary_codes=np.asarray(artifact["cam_binary_codes"], dtype=np.int8),
            alphas=np.asarray(artifact["alphas"], dtype=np.float32),
            bias=np.array([artifact["bias"]], dtype=np.float32),
            threshold=np.array([artifact["threshold"]], dtype=np.float32),
            method=np.array([method], dtype=object),
            method_title=np.array([HASH_METHOD_TITLES[method]], dtype=object),
            code_length=np.array([D_MODEL], dtype=np.int64),
            model_id=np.array([MODEL_ID], dtype=object),
            layer_index=np.array([LAYER_INDEX], dtype=np.int64),
            hidden_state_mode=np.array([HIDDEN_STATE_MODE], dtype=object),
            row_metadata_json=np.array([json.dumps(all_scope_meta[scope])], dtype=object),
            history_json=np.array([json.dumps(artifact["history"])], dtype=object),
            metrics_json=np.array([json.dumps(artifact["metrics"])], dtype=object),
            hash_matrix_path=np.array([artifact.get("hash_matrix_path")], dtype=object),
        )
        print("Saved", out_path)

if not SAVE_LEARNED_HASH_MATRICES:
    print(
        "Learned d × d matrices were not saved because SAVE_LEARNED_HASH_MATRICES=False. "
        "Set it to True before running the experiment to retain each dense matrix."
    )
